In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import requests

df_modelo = pd.read_csv("df_modelo_links.csv")

In [48]:
def url_valida(url):
    if not url.startswith("http"):
        return False
    try:
        resposta = requests.head(url, allow_redirects=True, timeout=5)
        return resposta.status_code == 200
    except:
        return False

df_modelo = df_modelo[df_modelo["titulo"].apply(url_valida)]
df_modelo = df_modelo[df_modelo["proximo_video"].apply(url_valida)]

print("Número de transições válidas:", len(df_modelo))

Número de transições válidas: 20


In [ ]:
le_titulo = LabelEncoder()
le_proximo = LabelEncoder()

df_modelo["titulo_encoded"] = le_titulo.fit_transform(df_modelo["titulo"])
df_modelo["proximo_encoded"] = le_proximo.fit_transform(df_modelo["proximo_video"])

X = df_modelo[["titulo_encoded"]]
y = df_modelo["proximo_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
print("Acurácia:", accuracy_score(y_test, y_pred))

Acurácia: 0.0


In [ ]:
def recomendar_top_n(video, modelo, le_titulo, le_proximo, n=3):
    if video not in le_titulo.classes_:
        return []
    
    video_enc = le_titulo.transform([video]).reshape(1, -1)
    
    probs = modelo.predict_proba(video_enc)[0]
    
    top_n_idx = probs.argsort()[::-1][:n]
    
    top_n_links = le_proximo.inverse_transform(top_n_idx)
    top_n_probs = probs[top_n_idx]
    
    return list(zip(top_n_links, top_n_probs))

video_atual = "https://www.youtube.com/watch?v=DbHssEeKEFs"
top_videos = recomendar_top_n(video_atual, modelo, le_titulo, le_proximo, n=3)

if top_videos:
    print("Top 3 próximos vídeos recomendados:")
    for link, prob in top_videos:
        print(f"{link} (probabilidade: {prob:.2f})")
else:
    print("Nenhum próximo vídeo encontrado para:", video_atual)

Top 3 próximos vídeos recomendados:
https://www.youtube.com/watch?v=KQO0po1LmXc (probabilidade: 0.62)
https://www.youtube.com/watch?v=GdGXPlDUYLY (probabilidade: 0.23)
https://www.youtube.com/watch?v=ayajTrETG48 (probabilidade: 0.12)


C:\Users\guilhermepinheiro-ie\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
